# Matched-training robustness study at 130M parameters

Runs `scripts/scale_study.py` on one GPU (A100 recommended, L4 works).

Every arm starts from the same public checkpoint (`state-spaces/mamba-130m-hf`) and gets the same data, optimiser, steps and seeds:
unconstrained vs. constrained, clean vs. adversarial fine-tuning, plus single-constraint ablations.
Every attack is re-optimised against each fine-tuned checkpoint.

Constrained arms come in two versions. `lipcal` uses constants calibrated to the checkpoint, so the model starts as almost the same function. `lip` uses the paper's constants, which change the pretrained function drastically.

**Runtime → Change runtime type → A100 GPU (or L4).** Run the cells in order. Results are written to Google Drive after every stage, so a disconnected session resumes where it stopped: just run the same cells again.

No Kaggle or Hugging Face token is needed: the checkpoint, tokenizer and WikiText-2 are public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type"
gpu = torch.cuda.get_device_name(0); PRESET = "a100" if "A100" in gpu else "l4"
print("GPU:", gpu, "| preset:", PRESET)

## 1. Output folder on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/lipmamba_scale_study"
import os; os.makedirs(OUT, exist_ok=True); os.environ["OUT"] = OUT; os.environ["PRESET"] = PRESET

## 2. Code

In [ ]:
%cd /content
# if this download fails, download the repository zip from the anonymous link and upload it here as repo.zip
!rm -rf LipMamba-ICLR27 repo.zip && wget -q -O repo.zip https://anonymous.4open.science/api/repo/LipMamba-ICLR27/zip && unzip -q -o repo.zip -d LipMamba-ICLR27 && (cd LipMamba-ICLR27 && [ -d src ] || mv */* . 2>/dev/null; true)
%cd /content/LipMamba-ICLR27
!pip -q install transformers datasets pytest
os.environ["PYTHONPATH"] = "/content/LipMamba-ICLR27/src"
!ls

## 3. Checks (about 2 minutes)
The first command confirms that the converted model computes the same function as the `transformers` Mamba implementation. The second runs the whole pipeline on a toy model.

In [ ]:
!python -m pytest -q tests/test_pretrained.py
!python scripts/scale_study.py run --model lip_at --tiny --out /content/smoke/lip_at_s0.json

## 4. Starting point and pilot (about 10 minutes)

**First command (no training):** perplexity and accuracy of every constrained variant at the start. `lipcal` should be close to `mamba (public checkpoint)`, which is about 26. If one variant "without ..." is close while `lipcal` is far, that constraint is the one doing the damage.

**Then the pilot:** 100 fine-tuning steps and a small attack run for `mamba` and `lipcal`. In both, `acc` should reach about 1.0.

Paste the whole output of this cell before starting section 5.

In [ ]:
!python scripts/scale_study.py diagnose --out $OUT/pilot/diagnose.json
!python scripts/scale_study.py run --model mamba  --seed 0 --preset $PRESET --steps 100 --lengths 8 --n-eval 16 --iters 10 --force --out $OUT/pilot/mamba_s0.json
!python scripts/scale_study.py run --model lipcal --seed 0 --preset $PRESET --steps 100 --lengths 8 --n-eval 16 --iters 10 --force --out $OUT/pilot/lipcal_s0.json

## 5. Main study: five arms, three seeds

`mamba_released`, `mamba`, `lipcal`, `mamba_at`, `lipcal_at`, 400 fine-tuning steps each. Finished arms are skipped when you re-run the cell, and an arm whose fine-tuning already finished resumes at the attacks. Each arm prints its own timing.

If the cell ends with a line starting `!!!!`, some runs failed: paste the first `Traceback` you find above it.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 1 2 -- --preset $PRESET

## 6. Ablations and paper constants
Run these when section 5 has finished, as far as your compute allows.

1. Four single-constraint ablations (seed 0), which answer the request for an ablation with attack success rate.
2. `lip`, `lip_at`: the paper's constants, same budget, three seeds.

In [ ]:
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 --models lipcal_no_sn lipcal_no_clamp lipcal_free_A lipcal_no_clip -- --preset $PRESET
!python scripts/scale_study.py all --dir $OUT/main --seeds 0 1 2 --models lip lip_at -- --preset $PRESET

## 7. Summary and download
Upload `lipmamba_scale_results.zip` (a few hundred kilobytes: JSON files only) to the chat.

In [ ]:
!python scripts/scale_study.py aggregate --dir $OUT/main
!cd $OUT && zip -qr /content/lipmamba_scale_results.zip main pilot -x '*.pt'
from google.colab import files; files.download('/content/lipmamba_scale_results.zip')